In [1]:
pip install cohere python-dotenv networkx

  Using cached python_dotenv-1.2.2-py3-none-any.whl.metadata (27 kB)
  Using cached pydantic-2.12.5-py3-none-any.whl.metadata (90 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached idna-3.11-py3-none-any.whl.metadata (8.4 kB)
  Using cached certifi-2026.2.25-py3-none-any.whl.metadata (2.5 kB)
  Using cached anyio-4.13.0-py3-none-any.whl.metadata (4.5 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached pydantic_core-2.41.5-cp312-cp312-macosx_11_0_arm64.whl.metadata (7.3 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
  Using cached click-8.3.1-py3-none-any.whl.metadata (2.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 28.7 MB/s  0:00:00
Using cached idna-3.11-py3-none-any.whl (71 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 51.

In [35]:
import cohere
import json
import networkx as nx
from dotenv import load_dotenv
import os
import time

load_dotenv()

api_key = os.getenv("COHERE_API_KEY")

if not api_key:
    raise ValueError("COHERE_API_KEY not found in environment. Check your .env file.")
print("API key loaded.")

API key loaded.


In [36]:
co = cohere.Client(api_key)

In [37]:
def chat_with_retry(max_retries=4, **kwargs):
    """Wraps co.chat with exponential backoff on TooManyRequestsError."""
    for attempt in range(max_retries):
        try:
            return co.chat(**kwargs)
        except Exception as e:
            if "TooManyRequests" in type(e).__name__ or "429" in str(e):
                wait = 15 * (2 ** attempt)  # 15s, 30s, 60s, 120s
                print(f"  Rate limited — waiting {wait}s (attempt {attempt + 1}/{max_retries})...")
                time.sleep(wait)
            else:
                raise  # re-raise non-rate-limit errors immediately
    raise RuntimeError("Max retries exceeded on Cohere rate limit.")

In [ ]:
# ── Dataset Configuration ──────────────────────────────────────────────────
# Change ACTIVE_DATASET to switch between datasets.
# Add new entries here when you upload the other two datasets.
DATASETS = {
    "wiki": {
        "data_path":     "data/wiki_corpus.json",
        "cache_path":    "data/graph_cache_wiki.json",
        "content_field": "clean_content",
        "title_field":   "title",
    },
    # "dataset2": {
    #     "data_path":     "data/dataset2.json",
    #     "cache_path":    "data/graph_cache_dataset2.json",
    #     "content_field": "text",    # adjust to match your field name
    #     "title_field":   "title",
    # },
    # "dataset3": {
    #     "data_path":     "data/dataset3.json",
    #     "cache_path":    "data/graph_cache_dataset3.json",
    #     "content_field": "text",
    #     "title_field":   "title",
    # },
}

ACTIVE_DATASET = "wiki"   # ← change this to switch datasets

cfg = DATASETS[ACTIVE_DATASET]
print(f"Active dataset: {ACTIVE_DATASET}")
print(f"  data  → {cfg['data_path']}")
print(f"  cache → {cfg['cache_path']}")

Active dataset: wiki
  data  → data/wiki_corpus.json
  cache → data/graph_cache_wiki.json


In [39]:
def extract_entities_llm(text):
    prompt = f"""Extract key entities from the text.

Return ONLY a valid JSON array of entity name strings. Example: ["Google", "London", "Alan Turing"]

Focus on: People, Organizations, Places, Technologies, Events.

TEXT:
{text[:3000]}
"""

    response = chat_with_retry(
        model="command-r7b-12-2024",
        message=prompt,
        temperature=0.2
    )

    raw = response.text.strip()
    if raw.startswith("```"):
        raw = raw.split("```")[1].lstrip("json").strip()
    try:
        entities = json.loads(raw)
        return [e for e in entities if isinstance(e, str)]
    except json.JSONDecodeError:
        return []

In [40]:
def extract_triplets(text):
    prompt = f"""Extract knowledge graph triplets from the text.

Return ONLY a valid JSON array of arrays, where each inner array is exactly 3 strings:
[["subject", "relation", "object"], ...]

Rules:
- Use short, clean entity names (2-4 words max)
- Use simple verb phrases as relations (e.g. "founded", "is part of", "developed")
- Return 5-15 triplets
- Do NOT include any explanation, just the JSON array

Text:
{text[:3000]}
"""

    response = chat_with_retry(
        model="command-r7b-12-2024",
        message=prompt,
        temperature=0
    )

    raw = response.text.strip()
    # Strip markdown code fences if present
    if raw.startswith("```"):
        raw = raw.split("```")[1].lstrip("json").strip()
    try:
        triplets = json.loads(raw)
        # Validate: keep only well-formed [subject, relation, object] entries
        return [t for t in triplets if isinstance(t, list) and len(t) == 3]
    except json.JSONDecodeError:
        return []

In [ ]:
with open(cfg["data_path"], "r", encoding="utf-8") as f:
    docs = json.load(f)

print(f"Loaded {len(docs)} docs from {cfg['data_path']}")
print("Keys:", list(docs[0].keys()))
print("Sample title:", docs[0][cfg["title_field"]])

def save_graph(graph, path):
    data = nx.node_link_data(graph)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2)
    print(f"Graph saved to {path}")

def load_graph(path):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    graph = nx.node_link_graph(data)
    print(f"Graph loaded from cache ({path})")
    return graph

def build_graph(documents, content_field, title_field, limit=None):
    """Build a knowledge graph from documents using LLM-extracted triplets.
    
    Args:
        documents:     list of dicts
        content_field: key containing the text to extract triplets from
        title_field:   key containing the document title (for progress display)
        limit:         optionally cap how many docs to process
    Returns:
        nx.Graph with edges labelled by relation
    """
    graph = nx.Graph()
    subset = documents[:limit] if limit else documents

    for i, doc in enumerate(subset):
        print(f"Processing doc {i+1}/{len(subset)}: {doc[title_field]}")
        triplets = extract_triplets(doc[content_field])

        for t in triplets:
            try:
                subject, relation, obj = t[0], t[1], t[2]
                graph.add_edge(subject.strip(), obj.strip(), relation=relation.strip())
            except (IndexError, TypeError):
                continue

        # Respect API rate limits
        time.sleep(0.5)

    return graph

# Load from cache if available, otherwise build and save
if os.path.exists(cfg["cache_path"]):
    G = load_graph(cfg["cache_path"])
else:
    # Increase limit (or remove it) once you're happy with the results
    G = build_graph(docs, cfg["content_field"], cfg["title_field"], limit=100)
    save_graph(G, cfg["cache_path"])

print(f"\nNodes: {len(G.nodes)}")
print(f"Edges: {len(G.edges)}")

Loaded 100 docs from data/wiki_corpus.json
Keys: ['title', 'content', 'links', 'clean_content']
Sample title: Artificial intelligence
Processing doc 1/100: Artificial intelligence
Processing doc 2/100: A* search algorithm
Processing doc 3/100: A.I. Artificial Intelligence
Processing doc 4/100: ACM Computing Classification System
Processing doc 5/100: Algorithm
Processing doc 6/100: Artificial intelligence
Processing doc 7/100: Bellman–Ford algorithm
Processing doc 8/100: Borůvka's algorithm
Processing doc 9/100: Dijkstra's algorithm
Processing doc 10/100: A.I. Artificial Intelligence (soundtrack)
Processing doc 11/100: Algorithm
Processing doc 12/100: Algorithmic efficiency
Processing doc 13/100: Analysis of algorithms
Processing doc 14/100: Computational intelligence
Processing doc 15/100: Algorithm aversion
Processing doc 16/100: Algorithm characterizations
Processing doc 17/100: Algorithm engineering
Processing doc 18/100: Algorithmic bias
Processing doc 19/100: Algorithmic composit

In [42]:
def traverse(graph, seeds, depth=2):
    visited = set()
    result = set()

    frontier = [(s, 0) for s in seeds]

    while frontier:
        node, d = frontier.pop()

        if node in visited or d > depth:
            continue

        visited.add(node)
        result.add(node)

        for neighbor in graph.neighbors(node):
            frontier.append((neighbor, d + 1))

    return result

In [43]:
def build_context(graph, nodes):
    context = []

    for n in nodes:
        for neighbor in graph.neighbors(n):
            relation = graph[n][neighbor].get("relation", "related_to")
            context.append(f"{n} -[{relation}]-> {neighbor}")

    return "\n".join(context)

In [44]:
def answer(query, context):
    prompt = f"""You are a GraphRAG assistant.

Use ONLY the knowledge graph below to answer the question.

GRAPH:
{context}

QUESTION:
{query}

Provide a clear, factual answer using the relationships shown above.
"""

    response = chat_with_retry(
        model="command-r7b-12-2024",
        message=prompt,
        temperature=0.3
    )

    return response.text

In [45]:
def get_seed_nodes(query, graph):
    """Find graph nodes that are relevant to the query.
    
    First tries direct substring matching against node names.
    Falls back to LLM entity extraction if no matches found.
    """
    query_lower = query.lower()
    seeds = []

    # Direct match: node name appears in query or query contains node name
    for node in graph.nodes:
        node_lower = str(node).lower()
        if node_lower in query_lower or query_lower in node_lower:
            seeds.append(node)

    # Fallback: use LLM to extract entities from query and match against graph
    if not seeds:
        entities = extract_entities_llm(query)
        for entity in entities:
            entity_lower = entity.lower()
            for node in graph.nodes:
                node_lower = str(node).lower()
                if entity_lower in node_lower or node_lower in entity_lower:
                    seeds.append(node)

    return list(set(seeds))[:5]  # deduplicate and cap at 5 seeds

In [46]:
def graphrag(query, graph=None, return_context=False):
    """Full GraphRAG pipeline: seed → traverse → context → answer.
    
    Args:
        query:          the user question
        graph:          nx.Graph to use (defaults to global G)
        return_context: if True, returns dict {"answer": ..., "context": ...}
                        instead of just the answer string
    """
    g = graph if graph is not None else G

    seeds = get_seed_nodes(query, g)
    if not seeds:
        empty = "No relevant nodes found in the graph for this query."
        return {"answer": empty, "context": ""} if return_context else empty

    subgraph_nodes = traverse(g, seeds)
    context = build_context(g, subgraph_nodes)

    if not context:
        empty = "Found seed nodes but no surrounding relationships to build context from."
        return {"answer": empty, "context": ""} if return_context else empty

    ans = answer(query, context)
    return {"answer": ans, "context": context} if return_context else ans

In [53]:
# ── Step 1: Generate test questions ────────────────────────────────────────
# Generates questions from a sample of the corpus and caches them so the
# same question set is reused across every RAG method your group tests.

QUESTIONS_CACHE = f"data/test_questions_{ACTIVE_DATASET}.json"
NUM_DOCS_TO_SAMPLE = 8   # sample more since off-topic docs return empty arrays
QUESTIONS_PER_DOC  = 3   # questions per doc  →  up to 8 × 3 = 24, but ~15 after filtering

def generate_questions_from_doc(doc_text, doc_title):
    prompt = f"""You are creating an evaluation dataset for a GraphRAG system.

GraphRAG stores knowledge as a graph of entities and relationships. It excels at questions
that require connecting multiple entities — NOT simple fact lookups like dates or names.

Given the text below, write {QUESTIONS_PER_DOC} questions that:
- Are about ARTIFICIAL INTELLIGENCE topics only (skip if the text is off-topic)
- Require reasoning about RELATIONSHIPS between entities (e.g. "How is X related to Y?",
  "What connects A and B?", "Which technologies are associated with X?")
- Cannot be answered with a single word or date — answers should describe connections
- Vary in the entities involved (don't ask about the same entity twice)

Bad examples (too simple, avoid these):
  - "Who founded OpenAI?" 
  - "What year was GPT-3 released?"

Good examples (relationship-focused):
  - "How is reinforcement learning related to robotics?"
  - "What is the relationship between transformer models and natural language processing?"
  - "How did Alan Turing's work influence the development of modern AI?"

If the text is NOT about AI, return an empty array: []

Return ONLY a valid JSON array of objects:
[{{"question": "...", "reference_answer": "..."}}]

Title: {doc_title}
Text: {doc_text[:2000]}
"""
    response = co.chat(model="command-r7b-12-2024", message=prompt, temperature=0.3)
    raw = response.text.strip()
    if raw.startswith("```"):
        raw = raw.split("```")[1].lstrip("json").strip()
    try:
        items = json.loads(raw)
        return [q for q in items if "question" in q and "reference_answer" in q]
    except json.JSONDecodeError:
        return []

if os.path.exists(QUESTIONS_CACHE):
    with open(QUESTIONS_CACHE, "r", encoding="utf-8") as f:
        test_questions = json.load(f)
    print(f"Loaded {len(test_questions)} cached test questions from {QUESTIONS_CACHE}")
else:
    import random

    AI_KEYWORDS = {
        "artificial intelligence", "machine learning", "deep learning", "neural network",
        "natural language", "computer vision", "robotics", "algorithm", "language model",
        "reinforcement learning", "transformer", "generative", "large language", "llm",
        "intelligent system", "automation", "cognition", "knowledge graph", "chatbot",
    }

    def is_ai_doc(doc):
        text = (doc.get(cfg["title_field"], "") + " " + doc.get(cfg["content_field"], "")[:300]).lower()
        return any(kw in text for kw in AI_KEYWORDS)

    ai_docs = [d for d in docs if is_ai_doc(d)]
    print(f"AI-relevant docs: {len(ai_docs)}/{len(docs)}")

    sample_docs = random.sample(ai_docs, min(NUM_DOCS_TO_SAMPLE, len(ai_docs)))
    test_questions = []

    for doc in sample_docs:
        print(f"Generating questions for: {doc[cfg['title_field']]}")
        qs = generate_questions_from_doc(doc[cfg["content_field"]], doc[cfg["title_field"]])
        test_questions.extend(qs)
        time.sleep(1)

    with open(QUESTIONS_CACHE, "w", encoding="utf-8") as f:
        json.dump(test_questions, f, indent=2, ensure_ascii=False)
    print(f"\nSaved {len(test_questions)} test questions to {QUESTIONS_CACHE}")

for i, q in enumerate(test_questions[:3], 1):
    print(f"\nQ{i}: {q['question']}")

Loaded 24 cached test questions from data/test_questions_wiki.json

Q1: How does the criss-cross algorithm differ from the simplex algorithm in terms of its approach to solving linear programming problems?

Q2: What is the significance of the Klee–Minty cube in the context of the criss-cross algorithm and the simplex algorithm?

Q3: How do the criss-cross algorithm and the simplex algorithm handle the Klee–Minty cube differently in terms of their average number of corner visits?


In [54]:
# ── Step 2: LLM Judge ──────────────────────────────────────────────────────
# Scores a single (question, context, answer) triple on 3 metrics.

def llm_judge(question, context, answer_text):
    """Score an answer using an LLM judge.

    Returns a dict with scores (1–5) for:
        faithfulness      – every claim is supported by the context
        answer_relevance  – answer directly addresses the question
        context_relevance – retrieved context was useful for the question

    A reasoning string is also returned for transparency.
    """
    prompt = f"""You are an expert evaluator for Retrieval-Augmented Generation (RAG) systems.

Score the answer below on three criteria using a 1–5 integer scale:

  1 = very poor  |  3 = acceptable  |  5 = excellent

Criteria:
- faithfulness:      Every claim in the answer is directly supported by the provided context.
                     Penalise heavily for hallucination or any statement not grounded in the context.
- answer_relevance:  The answer directly and completely addresses the question with a real answer.
                     IMPORTANT: If the answer says it "cannot answer", "no information", "not found in
                     the graph", or anything similar — that is a FAILED answer. Score it 1.
- context_relevance: The retrieved context actually contained information useful for answering
                     the question. If the context is off-topic or the answer says no info was found,
                     score this 1.

QUESTION:
{question}

RETRIEVED CONTEXT:
{context if context else "(no context retrieved)"}

ANSWER:
{answer_text}

Return ONLY a valid JSON object (no extra text):
{{
  "faithfulness": <1-5>,
  "answer_relevance": <1-5>,
  "context_relevance": <1-5>,
  "reasoning": "<one sentence explaining the scores>"
}}
"""
    response = chat_with_retry(model="command-r7b-12-2024", message=prompt, temperature=0)
    raw = response.text.strip()
    if raw.startswith("```"):
        raw = raw.split("```")[1].lstrip("json").strip()
    try:
        scores = json.loads(raw)
        # Clamp scores to valid range
        for key in ("faithfulness", "answer_relevance", "context_relevance"):
            scores[key] = max(1, min(5, int(scores.get(key, 1))))
        return scores
    except (json.JSONDecodeError, ValueError):
        return {"faithfulness": 0, "answer_relevance": 0, "context_relevance": 0,
                "reasoning": "parse error"}

# Quick smoke test
_test = llm_judge(
    "What is machine learning?",
    "Machine learning -[is part of]-> Artificial intelligence",
    "Machine learning is a subfield of artificial intelligence."
)
print("Judge smoke test:", _test)

Judge smoke test: {'faithfulness': 5, 'answer_relevance': 5, 'context_relevance': 5, 'reasoning': 'The answer is directly supported by the context, is relevant to the question, and the context is useful for answering the question.'}


In [56]:
# ── Step 3: Run evaluation ─────────────────────────────────────────────────
# Saves results after every question so a crash won't lose progress.
# Re-running this cell will skip already-completed questions automatically.

EVAL_RESULTS_PATH = f"data/eval_results_{ACTIVE_DATASET}.json"

# Resume from existing results if available
if os.path.exists(EVAL_RESULTS_PATH):
    with open(EVAL_RESULTS_PATH, "r", encoding="utf-8") as f:
        eval_results = json.load(f)
    done_questions = {r["question"] for r in eval_results}
    print(f"Resuming — {len(eval_results)} already done, {len(test_questions) - len(done_questions)} remaining.")
else:
    eval_results = []
    done_questions = set()

for i, item in enumerate(test_questions):
    question = item["question"]

    if question in done_questions:
        print(f"[{i+1}/{len(test_questions)}] Skipping (already done): {question[:60]}")
        continue

    print(f"[{i+1}/{len(test_questions)}] {question}")

    for attempt in range(3):
        try:
            result = graphrag(question, return_context=True)
            ans    = result["answer"]
            ctx    = result["context"]
            scores = llm_judge(question, ctx, ans)
            break
        except Exception as e:
            if "TooManyRequests" in type(e).__name__ or "429" in str(e):
                wait = 20 * (attempt + 1)  # 20s, 40s, 60s
                print(f"  Rate limited — waiting {wait}s (attempt {attempt + 1}/3)...")
                time.sleep(wait)
            else:
                raise
    else:
        print(f"  Skipping after 3 failed attempts.")
        continue

    eval_results.append({
        "question":          question,
        "reference_answer":  item.get("reference_answer") or item.get("reference", ""),
        "graphrag_answer":   ans,
        "context":           ctx,
        "faithfulness":      scores["faithfulness"],
        "answer_relevance":  scores["answer_relevance"],
        "context_relevance": scores["context_relevance"],
        "reasoning":         scores.get("reasoning", ""),
    })

    # Save after every question so progress is never lost on a crash
    with open(EVAL_RESULTS_PATH, "w", encoding="utf-8") as f:
        json.dump(eval_results, f, indent=2, ensure_ascii=False)

    time.sleep(4)  # 2 calls/question × 3s minimum per call; 4s gives safe headroom

print(f"\nDone. {len(eval_results)}/{len(test_questions)} results saved to {EVAL_RESULTS_PATH}")

# ── Summary table ───────────────────────────────────────────────────────────
metrics = ("faithfulness", "answer_relevance", "context_relevance")
averages = {m: sum(r[m] for r in eval_results) / len(eval_results) for m in metrics}

print(f"\n{'='*45}")
print(f"  GraphRAG Evaluation  —  dataset: {ACTIVE_DATASET}")
print(f"{'='*45}")
print(f"  Questions evaluated : {len(eval_results)}")
print(f"  Faithfulness        : {averages['faithfulness']:.2f} / 5")
print(f"  Answer Relevance    : {averages['answer_relevance']:.2f} / 5")
print(f"  Context Relevance   : {averages['context_relevance']:.2f} / 5")
avg_overall = sum(averages.values()) / len(averages)
print(f"  Overall             : {avg_overall:.2f} / 5")
print(f"{'='*45}")

# Per-question breakdown
print("\nPer-question scores (F=Faithfulness, A=Answer Rel., C=Context Rel.):")
print(f"{'#':<4} {'F':>3} {'A':>3} {'C':>3}  Question")
print("-" * 65)
for i, r in enumerate(eval_results, 1):
    q_preview = r["question"][:48] + "…" if len(r["question"]) > 48 else r["question"]
    print(f"{i:<4} {r['faithfulness']:>3} {r['answer_relevance']:>3} {r['context_relevance']:>3}  {q_preview}")

[1/24] How does the criss-cross algorithm differ from the simplex algorithm in terms of its approach to solving linear programming problems?
[2/24] What is the significance of the Klee–Minty cube in the context of the criss-cross algorithm and the simplex algorithm?
[3/24] How do the criss-cross algorithm and the simplex algorithm handle the Klee–Minty cube differently in terms of their average number of corner visits?
[4/24] How does the Bellman-Ford algorithm handle graphs with negative edge weights?
[5/24] What is the relationship between the Bellman-Ford algorithm and Dijkstra's algorithm?
[6/24] Who contributed to the development of the Bellman-Ford algorithm, and how is it named?
[7/24] How did the publication of 'Artificial Intelligence: A Modern Approach' influence the field of AI?
[8/24] What is the relationship between the topics covered in AIMA and the development of AI?
[9/24] How has the GitHub repository for AIMA contributed to the learning experience of AI students?
[10/